# Fine-Tuning a Protein Language Model for Subcellular Localization

In this lab, we'll compare two ways to adapt a protein language model, a specific type of neural network, for protein sequence classification.
After a model is pre-trained on a broad task, we can adapt it to a new task with additional training. This process is called **fine-tuning**.

Here, we'll fine-tune **ESM2 (8M parameters)** to predict **subcellular localization** from amino-acid sequence.

We will compare two strategies:

1. **Head-only fine-tuning**: freeze the ESM2 backbone and train only the final classification head.
2. **LoRA (PEFT)**: keep the backbone mostly frozen, but add small trainable adapters to selected attention layers.

The dataset is `proteinglm/localization_prediction`, which has 10 localization classes.
We evaluate both approaches with:

- **Accuracy**: overall fraction correct.
- **Macro-F1**: averages class-wise F1 equally, so minority classes matter more.

## Runtime note

This notebook is designed for **Google Colab with a GPU** (T4/L4 recommended).
Use `Runtime -> Change runtime type -> GPU` before running.
Default settings use small subsets so the notebook can finish quickly.

The next cell installs required packages for this session.


In [ ]:
%%capture
!pip install -q transformers datasets accelerate evaluate peft \
    scikit-learn xformers

## Import libraries and check the environment

We'll import all libraries up front and verify GPU availability.
We're using a lot of libraries here, but the new one to highligh is `transformers`.
This is a very popular library from HuggingFace that provides access to many open source models, mostly for natural language, but also for biology.

If you see an error, you likely need to take the steps described in the "Runtime note" section above.

In [ ]:
import os
import random
import time
from collections import Counter

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

cuda_ok = torch.cuda.is_available()
device = torch.device("cuda")
if not cuda_ok:
    raise RuntimeError("Make sure you selected a GPU as mentioned above!")

## Reproducibility and configuration

All experiment settings live in one `CONFIG` dictionary.
This makes it easy to run controlled comparisons and to modify one variable at a time.

Why these defaults?
- small subset sizes keep runtime short,
- `max_length=256` limits memory use,
- separate learning rates for head-only and LoRA give each method a fair starting point,
- explicit train/validation/test splits keep model selection and final evaluation separate.

We also set random seeds for Python, NumPy, and PyTorch so runs are as reproducible as possible.


In [ ]:
CONFIG = {
    "model_name": "facebook/esm2_t6_8M_UR50D",  # The model that we'll use
    "dataset_name": "proteinglm/localization_prediction",  # The dataset
    "train_n": 4000,  # Number of training examples
    "val_n": 500,  # Number of validation examples (from train split)
    "test_n": 500,  # Number of test examples
    "max_length": 256,  # Keep proteins short for speed
    "epochs": 10,  # Number of training epochs
    "lr_head": 2e-3,  # Learning rate for head-only
    "lr_lora": 2e-4,  # Learning rate for LoRA
    "train_batch_size": 8,  # Training batch size
    "eval_batch_size": 16,  # Validation/test batch size
    "seed": 0,  # Random seed for reproducibility
    "lora_r": 8,  # LoRA rank
    "lora_alpha": 16,  # LoRA scaling
    "lora_dropout": 0.05,  # LoRA dropout
}


def set_all_seeds(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


set_all_seeds(CONFIG["seed"])
print("Seed set to", CONFIG["seed"])

## Load the dataset, create train/validation/test splits, and tokenize

For this specific dataset (`proteinglm/localization_prediction`):
- `raw["train"]` is used to create the training and validation sets,
- `raw["validation"]` is used as the final test set.

This means:
- **training set**: for fitting model parameters,
- **validation set**: split from training data for model/hyperparameter decisions,
- **test set**: predefined split used only for final performance reporting.

Then we tokenize sequences from the `seq` field.

Important preprocessing choices:
- `truncation=True`: long sequences are cut to `max_length`,
- `padding='max_length'`: all examples have the same length for batching,
- fixed `max_length`: improves throughput and predictability on Colab GPUs.

We also print class counts so you can see whether labels are imbalanced.


In [ ]:
raw = load_dataset(CONFIG["dataset_name"])
print(raw)

if "test" not in raw:
    raise ValueError(
        "This notebook expects a predefined 'test' split for test data."
    )

train_full = raw["train"].shuffle(seed=CONFIG["seed"])
test_full = raw["test"].shuffle(seed=CONFIG["seed"])

# Validation is always carved from the training split.
train_val_n = min(CONFIG["train_n"] + CONFIG["val_n"], len(train_full))
train_val_subset = train_full.select(range(train_val_n))

train_n = min(CONFIG["train_n"], len(train_val_subset))
val_n = min(CONFIG["val_n"], len(train_val_subset) - train_n)

test_n = min(CONFIG["test_n"], len(test_full))

train_ds = train_val_subset.select(range(train_n))
val_ds = train_val_subset.select(range(train_n, train_n + val_n))
test_ds = test_full.select(range(test_n))

print(f"Using train split size: {len(train_ds)}")
print(f"Using validation split size: {len(val_ds)}")
print(f"Using test split size: {len(test_ds)}")
print("Test data source: predefined 'validation' split")

for split_name, split_ds in [
    ("train", train_ds),
    ("validation", val_ds),
    ("test", test_ds),
]:
    counts = Counter(split_ds["label"])
    print(f"Label counts in {split_name} subset:")
    for label_id, count in sorted(counts.items()):
        print(f"  class {label_id}: {count}")

sample_for_length = min(500, len(train_ds))
mean_len = float(
    np.mean([len(s) for s in train_ds.select(range(sample_for_length))["seq"]])
)
print(
    f"Average sequence length (sample of {sample_for_length} train examples): "
    f"{mean_len:.1f}"
)

Now we need to tokenize the protein sequences.
Tokenization is the process of mapping amino-acid characters to numerical IDs that the model can process.
This is the responsibility of the tokenizer.

Since we're using a pretrained model, it's important to use its matching tokenizer.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"], trust_remote_code=True
)


def tokenize_batch(batch):
    """Transform a batch of protein sequences into numeric tokens."""
    return tokenizer(
        batch["seq"],
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_length"],
    )


train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok = val_ds.map(tokenize_batch, batched=True)
test_tok = test_ds.map(tokenize_batch, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]


def keep_only_model_cols(ds):
    """Keep only the columns needed by the model and trainer."""
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds = ds.rename_column("label", "labels")
    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )
    return ds


train_tok = keep_only_model_cols(train_tok)
val_tok = keep_only_model_cols(val_tok)
test_tok = keep_only_model_cols(test_tok)

print("Tokenized columns (train):", train_tok.column_names)
print("Tokenized columns (validation):", val_tok.column_names)
print("Tokenized columns (test):", test_tok.column_names)

## Metrics helpers

We'll compute two complementary metrics:

- **Accuracy** for overall correctness.
- **Macro-F1** to give each class equal weight.

In localization tasks, class imbalance is common, so macro-F1 is often more informative than accuracy alone.
We'll also define a confusion matrix plot to inspect where errors happen.


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    """Compute accuracy and macro-F1 for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)[
        "accuracy"
    ]
    macro_f1 = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="macro",
    )["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}


def plot_confusion_matrix(y_true, y_pred, title, label_names=None):
    """Plot a confusion matrix for model predictions."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=label_names
    )
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Build models: head-only and LoRA

Both methods start from the same pretrained ESM2 checkpoint (weights that someone else trained and provides).
The key difference is **which parameters are trainable**.

- In **head-only**, only classifier parameters are updated.
- In **LoRA**, we inject trainable low-rank adapters into selected attention projections.

We print trainable parameter counts for transparency.
This is important: PEFT methods are mainly about reducing trainable parameters while keeping strong performance.


In [ ]:
NUM_LABELS = 10


def count_parameters(model):
    """Count the number of parameters in the model."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total


def print_trainable_summary(model, name):
    """Print the number of parameters that will be trained."""
    trainable, total = count_parameters(model)
    pct = 100 * trainable / total
    print(f"{name} trainable params: {trainable:,} / {total:,} ({pct:.2f}%)")


def build_head_only_model():
    """Build a sequence classifier with only the head trainable."""
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=NUM_LABELS,
    )

    for _, param in model.named_parameters():
        param.requires_grad = False

    head_keywords = ["classifier", "score", "classification_head"]
    matched = []
    for name, param in model.named_parameters():
        if any(k in name.lower() for k in head_keywords):
            param.requires_grad = True
            matched.append(name)

    if len(matched) == 0:
        raise ValueError(
            "Could not find classification head parameters to unfreeze. "
            "Inspect model.named_parameters() for head names."
        )

    print("Head-only trainable parameter names (sample):")
    for name in matched[:10]:
        print("  -", name)
    print_trainable_summary(model, "Head-only")
    return model


def find_lora_targets(model):
    """Infer likely attention modules for LoRA injection."""
    linear_names = [
        n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)
    ]

    exact = ["q_proj", "k_proj", "v_proj", "o_proj"]
    found_exact = []
    for key in exact:
        if any(n == key or n.endswith(f".{key}") for n in linear_names):
            found_exact.append(key)
    if found_exact:
        print("LoRA target modules (exact match):", found_exact)
        return found_exact

    fallback_checks = {
        "query": ["query"],
        "key": ["key"],
        "value": ["value"],
        "out": ["out", "output", "dense"],
    }

    found_fallback = []
    for name in linear_names:
        tail = name.split(".")[-1]
        lower = name.lower()
        if "attn" in lower or "attention" in lower:
            if any(
                any(tok in lower for tok in toks)
                for toks in fallback_checks.values()
            ):
                found_fallback.append(tail)

    found_fallback = sorted(set(found_fallback))
    if found_fallback:
        print("LoRA target modules (fallback match):", found_fallback)
        return found_fallback

    print("Could not find obvious attention projection names.")
    print("First 30 linear module names for debugging:")
    for name in linear_names[:30]:
        print("  -", name)
    raise ValueError("No LoRA target modules found automatically.")


def build_lora_model():
    """Build a PEFT LoRA sequence-classification model."""
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=NUM_LABELS,
    )
    targets = find_lora_targets(model)

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        target_modules=targets,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    print_trainable_summary(model, "LoRA")
    return model

## Training runner

This helper function wraps Hugging Face `Trainer` so both experiments use the same pipeline.

It trains on the training set, monitors the validation set during training, and then evaluates on both validation and test sets.

It returns:
- training output,
- validation metrics,
- test metrics,
- test predictions and labels,
- wall-clock runtime.


In [ ]:
def _training_args(run_name, lr, batch_train, batch_eval, epochs):
    args = {
        "output_dir": f"out/{run_name}",
        "per_device_train_batch_size": batch_train,
        "per_device_eval_batch_size": batch_eval,
        "learning_rate": lr,
        "num_train_epochs": epochs,
        "logging_steps": 25,
        "save_strategy": "no",
        "report_to": "none",
        "seed": CONFIG["seed"],
        "data_seed": CONFIG["seed"],
        "fp16": cuda_ok,
    }

    # Transformers renamed this argument in newer versions.
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        args["eval_strategy"] = "epoch"
    else:
        args["evaluation_strategy"] = "epoch"

    return TrainingArguments(**args)


def train_and_evaluate(model, run_name, lr, batch_train, batch_eval, epochs):
    """Train one model configuration and return validation/test metrics."""
    args = _training_args(run_name, lr, batch_train, batch_eval, epochs)

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_tok,
        "eval_dataset": val_tok,
        "compute_metrics": compute_metrics,
    }

    # `tokenizer` was renamed to `processing_class` in newer Transformers.
    trainer_init_args = Trainer.__init__.__code__.co_varnames
    if "processing_class" in trainer_init_args:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_init_args:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    start = time.time()
    train_result = trainer.train()
    runtime_s = time.time() - start

    val_metrics = trainer.evaluate(eval_dataset=val_tok)
    test_pred_out = trainer.predict(test_tok)
    test_preds = np.argmax(test_pred_out.predictions, axis=-1)
    test_labels = test_pred_out.label_ids
    test_metrics = test_pred_out.metrics

    if test_preds.shape[0] != test_labels.shape[0]:
        raise ValueError("Predictions and labels length mismatch.")

    return {
        "trainer": trainer,
        "train_result": train_result,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "runtime_s": runtime_s,
    }

## Experiment A: head-only fine-tuning

This is our baseline adaptation strategy.

Before running, make a hypothesis:
- Will training only the head be enough?
- Which metric (accuracy or macro-F1) do you expect to be stronger/weaker?


In [ ]:
head_model = build_head_only_model()
head_trainable, head_total = count_parameters(head_model)

results_head = train_and_evaluate(
    model=head_model,
    run_name="head_only",
    lr=CONFIG["lr_head"],
    batch_train=CONFIG["train_batch_size"],
    batch_eval=CONFIG["eval_batch_size"],
    epochs=CONFIG["epochs"],
)

print("Head-only validation metrics:")
for k, v in results_head["val_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

print()
print("Head-only test metrics:")
for k, v in results_head["test_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

## Experiment B: LoRA fine-tuning

Now we'll run LoRA with the same data split and evaluation protocol.

Focus on three questions:
- Does LoRA improve macro-F1 over head-only?
- How much extra runtime does LoRA require?
- How many trainable parameters does LoRA add relative to head-only?


In [ ]:
lora_model = build_lora_model()
lora_trainable, lora_total = count_parameters(lora_model)

results_lora = train_and_evaluate(
    model=lora_model,
    run_name="lora",
    lr=CONFIG["lr_lora"],
    batch_train=CONFIG["train_batch_size"],
    batch_eval=CONFIG["eval_batch_size"],
    epochs=CONFIG["epochs"],
)

print("LoRA validation metrics:")
for k, v in results_lora["val_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

print()
print("LoRA test metrics:")
for k, v in results_lora["test_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

## Compare both runs on the test set

The table below summarizes efficiency and final test-set performance side by side.

Interpretation prompts:
- Which method has the better **test macro-F1**?
- Is the performance gain (if any) worth the extra trainable parameters or runtime?
- Do validation and test metrics suggest overfitting?


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "run": "head_only",
            "trainable_params": head_trainable,
            "trainable_percent": 100 * head_trainable / head_total,
            "test_accuracy": results_head["test_metrics"].get(
                "test_accuracy", np.nan
            ),
            "test_macro_f1": results_head["test_metrics"].get(
                "test_macro_f1", np.nan
            ),
            "runtime_s": results_head["runtime_s"],
        },
        {
            "run": "lora",
            "trainable_params": lora_trainable,
            "trainable_percent": 100 * lora_trainable / lora_total,
            "test_accuracy": results_lora["test_metrics"].get(
                "test_accuracy", np.nan
            ),
            "test_macro_f1": results_lora["test_metrics"].get(
                "test_macro_f1", np.nan
            ),
            "runtime_s": results_lora["runtime_s"],
        },
    ]
)
comparison

## Test-set confusion matrices

Confusion matrices reveal *which* classes are confused, not just how many examples are correct.

Use these plots to discuss:
- classes that are consistently difficult,
- whether LoRA helps specific minority classes,
- whether improvements in macro-F1 align with visible per-class gains.


In [ ]:
label_names = [
    "Nucleus",
    "Cytoplasm",
    "Extracellular",
    "Mitochondrion",
    "Cell membrane",
    "Endoplasmic reticulum",
    "Plastid",
    "Golgi apparatus",
    "Lysosome/Vacuole",
    "Peroxisome",
]

plot_confusion_matrix(
    results_head["test_labels"],
    results_head["test_preds"],
    title="Head-only confusion matrix (test set)",
    label_names=label_names,
)

plot_confusion_matrix(
    results_lora["test_labels"],
    results_lora["test_preds"],
    title="LoRA confusion matrix (test set)",
    label_names=label_names,
)